# Spatial transcriptomics of human skin wound healing: open notebook

**ENS de Lyon, M2 NGS practicals.** Data: Visium (GEO [GSE241124](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE241124)) and scRNA-seq (GEO [GSE241132](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE241132)) from *Spatiotemporal single-cell roadmap of human skin wound healing* (Liu et al., Cell Stem Cell 2025). We use one donor at four timepoints: healthy skin, and wound biopsies at day 1, 7 and 30.

### What this notebook is
A **guide**, not an exercise sheet. It lists the steps of the analysis and the questions we want to discuss. **You write most of the code**, and you are expected to **explore the tools** (`scanpy`, `squidpy`, `spatialdata`, `spatialdata_plot`...): read their documentation, try things, look at what objects contain. We start with one sample at a time and put them together later.

### How we work
- We go through the parts together. For each part: read the *Goal*, discuss, then try in the empty cells.
- **Hints** are folded. Open one only after you have tried. Hints with a full solution exist only for the parts that are easy to get stuck on for technical reasons (files, data structures, known software traps), **not** for the analysis choices (filtering, normalisation, clustering...): those are yours.
- A few **custom functions** are already written. Each one comes with an explanation of why it exists. Read them.
- Write your **choices and reasons** in the notes cells. This matters more than the code.

### What you deliver at the end
**Your own notebook**, answering a question you chose, with justified choices, figures and a discussion (see the last section).

## 0. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize
import seaborn as sns

import scanpy as sc
import squidpy as sq
import anndata as ad
import spatialdata as sd
import spatialdata_io
import spatialdata_plot
from spatialdata.models import TableModel
from scipy.stats import spearmanr

sns.set_style("whitegrid")

In [ ]:
# Paths: edit BASE, everything else is derived from it.
BASE = "/path/to/course/folder"      # <-- EDIT: the folder that contains the `data/` directory

DATA        = f"{BASE}/data/GSE241124_RAW"              # Visium: one Space Ranger folder per sample
SC_DATA     = f"{BASE}/data/GSE241132_RAW"              # scRNA-seq: one folder per sample
PROC        = f"{BASE}/data/processed_data"             # instructors' results (safety net, Cell2location)
MARKERS_CSV = f"{BASE}/data/paper_marker_genes.csv"     # marker genes reported in the paper
PAPER_META  = f"{SC_DATA}/GSE241132_cell_metadata.txt"  # per-cell annotation published by the authors
Q05_FILE    = f"{PROC}/q05_cell_abundance.csv"          # Cell2location output (computed by the instructors)

MY = f"{BASE}/my_results"                               # your own checkpoints
os.makedirs(MY, exist_ok=True)

SAMPLES    = ["Skin", "Wound1", "Wound7", "Wound30"]    # the 4 Visium sections
SC_SAMPLES = ["D0", "D1", "D7", "D30"]                  # the same 4 timepoints in scRNA-seq

## 1. Meet the data: one sample at a time

**Goal.** Load one Visium section and understand what is inside before analysing anything.

**To discuss together:**

- What is a Visium *spot*? How many cells can be in one?
- What do the four sections represent? What can (and cannot) we conclude comparing them, given that they come from one donor?
- Where are the counts, the spot coordinates and the tissue image stored in the object?

In [ ]:
# Start here: spatialdata_io reads the output folder of Space Ranger into a SpatialData object.
sdata = spatialdata_io.visium(f"{DATA}/Skin", dataset_id="Skin")
sdata

In [ ]:
# Explore: what are the elements of sdata? what is in sdata['table']  (.X, .obs, .var, .obsm)?
# How many spots, how many genes? What does one row of the coordinates mean?

*What did you find? (double-click to edit):* 

Now do the same for the **three other samples**. How would you store four SpatialData objects so that you can handle them together?

In [ ]:
# load Wound1, Wound7, Wound30

<details>
<summary><b>Hint: loading several samples</b> (try first, open if stuck)</summary>

A dictionary with one entry per sample is convenient: the sample name is the key.

```python
sdatas = {s: spatialdata_io.visium(f"{DATA}/{s}", dataset_id=s) for s in SAMPLES}
```

</details>

**Look at the tissue.** The image and the spots live in separate elements of the SpatialData object, and you need `spatialdata_plot` to draw them together. Try for *one* sample first (print `sdata` to find the names of the elements), then see the hint.

In [ ]:
# draw the tissue image of one sample, then the spots coloured by something (a gene, total counts...)

<details>
<summary><b>Hint: drawing a sample with spatialdata_plot</b> (try first, open if stuck)</summary>

The image, the spots (`shapes`) and the table are separate elements. `render_shapes` colours the spots with a column of the table (an `obs` column or a gene). The coordinate system `<sample>_downscaled_hires` matches the high-resolution image.

```python
s = "Skin"
(sdatas[s].pl.render_images(f"{s}_hires_image")
          .pl.render_shapes(s, color="total_counts", table_name="table", method="matplotlib")
          .pl.show(coordinate_systems=f"{s}_downscaled_hires", title=s))
# (total_counts does not exist yet: it is a QC metric, computed in the next section. Try a gene first, e.g. color="KRT17")
```

</details>

## 2. Quality control, sample by sample

**Goal.** Look at the quality of each section, and decide what to filter.

**To discuss together:**

- Which QC metrics would you compute for a spatial spot? Which genes should count as mitochondrial and why does it matter?
- Look at the distributions, then at the *same metrics on the tissue*. Is low quality random in space, or does it follow tissue structures?
- In intact tissue (not dissociated cells), what does a high mitochondrial fraction mean? Would you filter on it?
- Are the four sections alike? Would you use the same thresholds for all of them?

In [ ]:
# compute the QC metrics for one sample, look at them (violin plots) and on the tissue

<details>
<summary><b>Hint: computing QC metrics</b> (try first, open if stuck)</summary>

`scanpy` computes them for you. The tricky part is telling it which genes are mitochondrial. Compute them **directly on `sdata['table']`**: the metrics then live inside the SpatialData object, which lets you draw them on the tissue.

```python
table = sdatas["Skin"]["table"]
table.var["mt"] = table.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(table, qc_vars=["mt"], percent_top=None, inplace=True)
table.obs.head()     # total_counts, n_genes_by_counts, pct_counts_mt
```

</details>

Repeat for all samples. Do you write a loop, or a function? Why?

In [ ]:
# QC for the four samples, and compare them

*What do you see? Which thresholds will you use, and why? (double-click to edit):* 

In [ ]:
# apply your filters (spots) sample by sample

<details>
<summary><b>Hint: filtering spots and putting the filtered table back</b> (try first, open if stuck)</summary>

`scanpy` filters in a copy. To keep the tissue images and spot shapes working, put the filtered table back in the SpatialData object. Keep `.copy()`: a view cannot be stored.

```python
for s, sdata in sdatas.items():
    table = sdata["table"]
    keep = table.obs["n_genes_by_counts"] > 100          # YOUR threshold(s): one per sample if you can justify it
    print(s, "spots removed:", (~keep).sum(), "of", table.n_obs)
    sdata.tables["table"] = table[keep].copy()
```

</details>

## 3. Putting the samples together

Until now each sample lived in its own object. We would like **all samples in one object**, to analyse them jointly (same clusters, same UMAP). Try to do it before opening the hint. Think about what must be merged: the count tables, but also the images and the spot coordinates.

In [ ]:
# put the four samples together in one SpatialData object

<details>
<summary><b>Hint: merging the samples</b> (try first, open if stuck)</summary>

`spatialdata` can merge several SpatialData objects, keeping images, shapes and tables. `concatenate_tables=True` also stacks the count tables into one table. `spatialdata_io` stores the sample name in the column `region` of the table: we copy it to a column called `sample`, a clearer name.

```python
sdata_all = sd.concatenate(list(sdatas.values()), concatenate_tables=True)
adata = sdata_all["table"]
adata.obs["sample"] = adata.obs["region"].astype(str)
print(adata.shape)
adata.obs["sample"].value_counts()
```

</details>

### A custom function to draw on the tissue
Drawing one gene on one section already takes four chained calls, and we will want to look at many genes and many variables on the **four sections side by side**. We would copy-paste the same lines dozens of times, which is how errors enter an analysis. So we write the drawing code **once**, in a function.

Read it: it loops over the samples, chooses a colormap suited to the type of variable (continuous or categorical), and optionally forces a **common colour scale** (`vmax`) so that the panels are comparable.

In [ ]:
def plot_on_tissue(sdata, color, samples=SAMPLES, vmax=None, cmap="viridis"):
    """Draw a gene or a column of sdata["table"].obs on the tissue sections, one panel per sample.
    vmax: if given, all panels share the colour scale [0, vmax] (makes timepoints comparable)."""
    table = sdata["table"]
    categorical = color in table.obs and str(table.obs[color].dtype) == "category"
    fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 5), squeeze=False)
    for ax, s in zip(axes[0], samples):
        opts = dict(color=color, table_name="table", method="matplotlib")
        if not categorical:
            opts["cmap"] = cmap                                  # categories get their own colours
        if vmax is not None:
            opts["norm"] = Normalize(vmin=0, vmax=vmax, clip=True)
        (sdata.pl.render_images(f"{s}_hires_image")
              .pl.render_shapes(s, **opts)
              .pl.show(coordinate_systems=f"{s}_downscaled_hires", ax=ax, title=s))
    fig.suptitle(color)
    fig.tight_layout()
    plt.show()

In [ ]:
# try it: QC metrics on the four sections

### Filtering genes

**Goal.** Decide which genes to remove, collectively across the four samples.

**To discuss together:**

- Which genes are detected in almost no spot? Why remove them?
- The paper removes *MALAT1*, mitochondrial genes and hemoglobin genes from the expression matrix. Why these? What do they have in common?
- Do you agree with removing them? What could go wrong?

In [ ]:
# gene filtering

<details>
<summary><b>Hint: hemoglobin genes: a trap</b> (try first, open if stuck)</summary>

Do **not** select hemoglobin genes with the prefix `HB`: you would also remove HBEGF (a real wound-healing growth factor), HBP1 and HBS1L. Use the explicit list. After filtering, store the new table in the object (`sdata_all.tables["table"] = adata`).

```python
HEMOGLOBIN = ["HBA1", "HBA2", "HBB", "HBD", "HBE1", "HBG1", "HBG2", "HBM", "HBQ1", "HBZ"]
drop = adata.var_names.str.startswith("MT-") | adata.var_names.isin(HEMOGLOBIN + ["MALAT1"])
adata = adata[:, ~drop].copy()
sc.pp.filter_genes(adata, min_cells=3)
sdata_all.tables["table"] = adata          # keep the SpatialData object in sync
```

</details>

*Your gene filtering choices (double-click to edit):* 

## 4. Normalisation, dimensionality reduction, clustering

**Goal.** Go from counts to a map of spot types.

**To discuss together:**

- Why normalise? Spots have very different total counts: what would happen if we did not?
- Why select variable genes? How many genes, how many PCs? How could you decide?
- Does the UMAP separate spots by *sample*, by *cluster*, or both?
- How many clusters is right? What does the *resolution* control?
- Do your clusters correspond to anatomy when drawn on the tissue (epidermis, dermis, wound bed)?

In [ ]:
# the standard single-cell recipe, applied to the spots. Your choices.

<details>
<summary><b>Hint: practical traps (not choices)</b> (try first, open if stuck)</summary>

Three things that silently go wrong with scanpy: (1) later we need the **raw counts** (Cell2location), keep them in a layer *before* normalising; (2) `neighbors` and `umap` overwrite each other's results unless you give a `key_added`/`neighbors_key`; (3) `sc.tl.leiden` has two implementations: the default (`leidenalg`) and `igraph`, which can give a different number of clusters from the same data. Know which one you use, and say it in your notes.

```python
adata.layers["counts"] = adata.X.copy()                 # raw counts, before normalisation
...
sc.tl.leiden(adata, resolution=0.5, flavor="igraph", n_iterations=2, directed=False)
```

</details>

In [ ]:
# draw your clusters on the tissue

*Your choices (genes, PCs, resolution) and why (double-click to edit):* 

## 5. Batch effects: correct or not?

**Goal.** Decide whether the four sections need to be integrated.

**To discuss together:**

- Colour your UMAP by sample. Is the separation technical (batch) or biological (time after wounding)?
- All four sections come from the **same donor**, each at a different time. Batch and biology are confounded: what could a correction erase?
- With four donors per timepoint, would you decide differently?
- How would you tell whether a correction helped or damaged the signal?

In [ ]:
# optional: try a correction method (Harmony, BBKNN, Scanorama...) and compare with your uncorrected result

<details>
<summary><b>Hint: Harmony: a known software trap</b> (try first, open if stuck)</summary>

`sc.external.pp.harmony_integrate` is currently broken with recent `harmonypy` versions (a transposition error). Call `harmonypy` directly. Give the new neighbour graph and UMAP their own keys, so you do not overwrite your uncorrected ones.

```python
import harmonypy
ho = harmonypy.run_harmony(adata.obsm["X_pca"], adata.obs, ["sample"])
adata.obsm["X_pca_harmony"] = ho.Z_corr
sc.pp.neighbors(adata, use_rep="X_pca_harmony", key_added="harmony")
sc.tl.umap(adata, neighbors_key="harmony")
adata.obsm["X_umap_harmony"] = adata.obsm["X_umap"].copy()   # store it: the next umap call overwrites X_umap
sc.pl.embedding(adata, basis="X_umap_harmony", color="sample")
```

</details>

*Your decision on batch correction and why (double-click to edit):* 

## 6. Where is the wound?

**Goal.** Use known genes to read the tissue before any annotation.

**To discuss together:**

- KRT6A, KRT16, KRT17 and S100A8 are described as wound-edge markers. Where and when do they light up?
- Does the pattern follow your clusters? Which cluster would you call the wound edge?
- A gene can be high in a broad region and still be a poor marker of one cell state. What makes a good marker?

In [ ]:
# genes on the four sections

<details>
<summary><b>Hint: saving a checkpoint and coming back to it</b> (try first, open if stuck)</summary>

Writing the SpatialData object lets you restart from here. Always write to a **new** path. If you reload and get an error about the table not being linked to the spots, rebuild the link (the information is stored in `adata.uns["spatialdata_attrs"]`, and does not always survive copies and saves).

```python
sdata_all.write(f"{MY}/spatial_clustered.zarr", overwrite=True)
sdata_all = sd.read_zarr(f"{MY}/spatial_clustered.zarr")      # to restart later

# if the link is lost (e.g. after saving an h5ad and reloading it):
adata = sdata_all["table"]
adata.uns.pop("spatialdata_attrs", None)
adata = TableModel.parse(adata, region=SAMPLES, region_key="region", instance_key="spot_id")
sdata_all.tables["table"] = adata
```

</details>

## 7. The single-cell reference

**Goal.** Analyse the scRNA-seq data of the same wound, to know *which cell populations* exist.

**To discuss together:**

- Same questions as before, in another technology: which QC, which thresholds? What is a doublet, and why is it a problem for annotation?
- Here a high mitochondrial fraction *does* mean something. What?
- How do you choose the number of clusters, and when is a cluster 'a cell type'?
- What does annotating a cluster really mean, and how sure are you of each label?

In [ ]:
# load the 4 timepoints of the donor PWH27 (folders PWH27D0, PWH27D1, PWH27D7, PWH27D30)

<details>
<summary><b>Hint: reading and combining the timepoints</b> (try first, open if stuck)</summary>

Each timepoint is a 10x matrix folder. The barcodes are not unique across timepoints (the same sequence can appear in two samples), so we append the timepoint to the barcode before concatenating.

```python
parts = []
for tp in SC_SAMPLES:
    a = sc.read_10x_mtx(f"{SC_DATA}/PWH27{tp}")
    a.var_names_make_unique()
    a.obs["sample"] = tp
    a.obs_names = [f"{bc}-{tp}" for bc in a.obs_names]
    parts.append(a)
adata_sc = ad.concat(parts, join="inner")
```

</details>

In [ ]:
# QC, filtering, doublets, normalisation, PCA, UMAP, clustering

<details>
<summary><b>Hint: two names to know</b> (try first, open if stuck)</summary>

Doublets are detected with `sc.pp.scrublet`, one sample at a time (`batch_key`). To choose the clustering resolution on data rather than by eye, you can try **RESOLUTE** (`import resolute as rs`; `rs.run_resolute(...)`): it scores many resolutions. Use it as an input to the discussion, not as the answer.

</details>

*Your choices and reasons (double-click to edit):* 

### Annotating the single-cell clusters

**Goal.** Name the clusters from their marker genes.

**To discuss together:**

- Which genes distinguish each cluster? Is a differentially expressed gene always a *marker*?
- The paper provides marker genes per cell type. Do they agree with your data?
- Do you stop at major cell types (keratinocytes, fibroblasts, immune cells...) or go further to cell states? What does each level allow you to say later?

In [ ]:
# differential expression, markers, then a name for each cluster

<details>
<summary><b>Hint: the paper's markers as a dictionary</b> (try first, open if stuck)</summary>

`paper_marker_genes.csv` lists the marker genes by cell type and dataset. A dotplot needs a dictionary {name: [genes]}, and fails if a gene is missing from your data: filter them first.

```python
marker_table = pd.read_csv(MARKERS_CSV)
print(marker_table.columns.tolist())

sc_markers = {}
for _, row in marker_table.query("dataset == 'scRNA-seq' and granularity == 'major'").iterrows():
    genes = [g for g in row["marker_genes"].split(";") if g in adata_sc.var_names]
    if genes:
        sc_markers[row["cell_type"]] = genes
sc.pl.dotplot(adata_sc, sc_markers, groupby="leiden", standard_scale="var")
```

</details>

<details>
<summary><b>Hint: optional: compare with the authors' own annotation</b> (try first, open if stuck)</summary>

The authors published a label for each cell. It is not 'the truth', but a useful second opinion. Their barcodes have another format than ours (`PWH27D0_AAAC...-1` versus `AAAC...-1-D0`), hence a small function to convert ours.

```python
paper_meta = pd.read_csv(PAPER_META, sep="\t", index_col=0)

def to_paper_barcode(barcode, donor="PWH27"):
    core, tp = barcode.rsplit("-", 1)          # "AAAC...-1-D0"  ->  "AAAC...-1", "D0"
    return f"{donor}{tp}_{core}"               # "PWH27D0_AAAC...-1"

keys = [to_paper_barcode(bc) for bc in adata_sc.obs_names]
adata_sc.obs["paper_celltype"] = pd.Series(keys, index=adata_sc.obs_names).map(paper_meta["newMainCellTypes"])
print(adata_sc.obs["paper_celltype"].notna().mean(), "of the cells matched")
pd.crosstab(adata_sc.obs["leiden"], adata_sc.obs["paper_celltype"])
```

</details>

*Your annotation, and your confidence in each label (double-click to edit):* 

## 8. Annotating the spatial clusters

**Goal.** Give a biological meaning to the clusters of spots.

**To discuss together:**

- A spot is a mixture of several cells. Can a cluster of spots be 'one cell type'? What would you call it (a tissue compartment, a niche)?
- Which genes characterise each cluster? Do the paper's spatial markers help?
- Do your names make sense on the tissue and on the H&E image?
- Which clusters are shared by the four sections and which are specific to a timepoint?

In [ ]:
# markers of the spot clusters, then name them (e.g. in adata.obs['spot_type'])

<details>
<summary><b>Hint: a trap with the paper's markers</b> (try first, open if stuck)</summary>

The same dictionary trick as for the cells, with the `ST-seq` entries. After your gene filtering, some markers may no longer be in the table (for example if you removed them): the dotplot would fail, so filter the list and *print* what is missing.

```python
st_markers = {}
for _, row in marker_table.query("dataset == 'ST-seq' and granularity == 'major'").iterrows():
    genes = row["marker_genes"].split(";")
    present = [g for g in genes if g in adata.var_names]
    if len(present) < len(genes):
        print(row["cell_type"], "missing:", sorted(set(genes) - set(present)))
    if present:
        st_markers[row["cell_type"]] = present
sc.pl.dotplot(adata, st_markers, groupby="leiden", standard_scale="var")
```

</details>

*Your spot annotation (double-click to edit):* 

## 9. Deconvolution: which cells are in each spot?

A spot contains several cells of several types. **Deconvolution** estimates, for each spot, how many cells of each type it contains, using the single-cell data as a dictionary. It is the **transfer of the annotation from the single-cell reference to the spatial data**.

We use **Cell2location**, a Bayesian model in two steps: (1) from the single-cell reference, learn an average expression *signature* for each cell state; (2) explain the counts of every spot as a combination of these signatures. The output is a table *spots x cell states* with an estimated abundance. We use `q05`, a conservative estimate (the 5% quantile of the posterior). Details and the quote from the paper: `q05_cell_abundance_explained.md`.

The training is long, so the instructors ran it. You can run it yourself if you want (ask for the script `run_cell2location.py`).

**Goal.** Understand what the result means before using it.

**To discuss together:**

- The reference has 17 fine cell states, defined by the instructors. What happens if a state present in the tissue is missing from the reference?
- The model returns an *abundance* (cells per spot), not a probability. Why is that suited to Visium spots?
- Errors in the single-cell annotation propagate to the spatial maps. How?
- What would you check to trust the result?

### A custom function to read the result
The instructors' table is indexed by *their* spots. Your table was filtered differently (your thresholds, your choices), so the spots do not match one to one, and a naive `adata.obs["x"] = q05["x"].values` would silently put **the wrong numbers on the wrong spots**. The function below matches the spots by **barcode + sample**, keeps the spots found in both tables, and adds one `c2l_<state>` column per cell state.

In [ ]:
def add_c2l(adata, q05_file=Q05_FILE):
    """Add the Cell2location abundances (q05) to adata.obs, matching spots by barcode and sample.
    Returns (adata restricted to the spots found in the table, q05 aligned to it)."""
    q05 = pd.read_csv(q05_file, index_col=0)
    barcode = r"([ACGT]{16}-\d)"
    sample = "(" + "|".join(SAMPLES) + ")"
    q_key = q05.index.str.extract(barcode)[0] + "|" + q05.index.str.extract(sample)[0]
    a_key = adata.obs_names.str.extract(barcode)[0].values + "|" + adata.obs["sample"].astype(str).values
    q05.index = q_key.values
    keep = pd.Series(a_key).isin(q05.index).values
    print(f"{keep.sum()} / {adata.n_obs} spots have an estimate; the others are dropped.")
    adata = adata[keep].copy()
    q05 = q05.loc[a_key[keep]]
    q05.index = adata.obs_names
    for state in q05.columns:
        adata.obs[f"c2l_{state}"] = q05[state].values
    adata.obs["dominant_state"] = q05.idxmax(axis=1).astype("category")
    return adata, q05

In [ ]:
adata, q05 = add_c2l(adata)
sdata_all.tables["table"] = adata          # keep the SpatialData object in sync (needed for plot_on_tissue)
print(q05.shape)
print(list(q05.columns))
q05.head()

<details>
<summary><b>Hint: if the table cannot be stored back</b> (try first, open if stuck)</summary>

Subsetting the table can lose the link between the table and the spots. See the hint of section 6 (`TableModel.parse`).

</details>

## 10. Reading the results: the interesting part

Each spot now has an estimated number of cells for each of 17 states (`q05`, and one `c2l_<state>` column per state in `adata.obs`). This is where spatial transcriptomics becomes powerful: **we know where each cell state is, at which time, and next to whom**, which single-cell data alone cannot tell.

Pick **at least three** questions below, or invent your own (the best option). For each: state a hypothesis *before* you plot, make the figure, and discuss what it shows and what it does not.

1. **When?** How do cell states change along healing? Which appear, peak, disappear?
2. **Where?** In which tissue compartment (your spot clusters) is each state found?
3. **Next to whom?** Which states are found together, which avoid each other? Does it change with time?
4. **Is it consistent?** Do the proportions in space and in the single-cell data agree? If not, why?
5. **Can I trust it?** Take one state and test the prediction against an independent marker gene. Is the gene specific enough?
6. **Your own question.**

Tools you may want to explore: `plot_on_tissue`, `pandas` (`groupby`, `crosstab`), `seaborn` (`heatmap`, `clustermap`), `scipy.stats.spearmanr`, and `squidpy` (`sq.gr.spatial_neighbors`, `sq.gr.nhood_enrichment`, `sq.gr.spatial_autocorr`).

In [ ]:
# Question 1: ...

In [ ]:
# Question 2: ...

In [ ]:
# Question 3: ...

<details>
<summary><b>Hint: squidpy: the spatial graph (a technical point)</b> (try first, open if stuck)</summary>

Spatial statistics need a graph of neighbouring spots, built **one section at a time** (coordinates of different sections are unrelated). Visium spots sit on a hexagonal grid: 6 neighbours.

```python
a = adata[adata.obs["sample"] == "Wound7"].copy()
sq.gr.spatial_neighbors(a, coord_type="grid", n_neighs=6, spatial_key="spatial")
sq.gr.nhood_enrichment(a, cluster_key="spot_type", seed=0)       # needs a categorical column, e.g. your spot_type
sq.pl.nhood_enrichment(a, cluster_key="spot_type")
```

</details>

<details>
<summary><b>Hint: comparing space and single-cell proportions</b> (try first, open if stuck)</summary>

Both tables need the same cell-state names. Your own single-cell annotation may be coarser: compare at the level you can, and say so.

</details>

*What did you learn? What are the limits? (double-click to edit):* 

## 11. Your deliverable

At the end of the course, hand in **your own notebook**, built from this one but written by you. It should be readable by someone who was not in the room.

**It must contain**
- A **question** you chose (from section 10 or your own), stated at the top.
- The analysis steps that lead to it, with **your choices explained** (thresholds, normalisation, clustering, annotation, batch correction or not).
- **Figures** that answer the question, with titles and legends that make sense alone.
- A **discussion**: what you can conclude, what you cannot, your assumptions, what you would do with more time or data.
- A notebook that **runs from top to bottom** (Kernel > Restart & Run All).

**We do not expect** perfect code or every tool of the practical. A simple analysis that is well argued beats a sophisticated one nobody can explain.